# Pipeline

**Yuxi Zhang** · Oct 25, 2025 · [@yuxi-research](https://github.com/yuxi-research)

This notebook pipelines all processes.


In [ ]:
re_preprocess = True # Set to True to re-run all data preprocessing notebooks from scratch
re_calibrate = True # Set to True to re-run all calibration notebooks from scratch

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import datetime as dt
from pathlib import Path
import papermill as pm

print(f"Running pipeline at {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Set farm name to run

In [ ]:
# Farm options: Amaroo, Dinyah, Willora, Llara, Nowley, Oodnadatta
farm_names = ['Amaroo', 'Dinyah', 'Llara', 'Nowley', 'Oodnadatta']

In [ ]:
# GADI Server Job Array support. This does not affect local runs where SITE_IDX is set to -99 by default.
PBS_ARRAY_INDEX = int(os.environ.get("PBS_ARRAY_INDEX", "-99"))
print("PBS_ARRAY_INDEX =", PBS_ARRAY_INDEX)

# Constrain the farm list to only the specified farm
if PBS_ARRAY_INDEX >= 0:
    farm_name = farm_names[PBS_ARRAY_INDEX]
else:
    farm_name = "Amaroo"  # Change farm name here for local runs


In [ ]:
##### Important: Make sure to restart the kernel after changing the farm name #####
os.environ['FARM_NAME'] = farm_name
os.environ['OUT_ROOT'] = f'pipeline_{os.environ["FARM_NAME"]}'


In [ ]:
# Path settings
from global_settings import ROOT, ProjectPaths
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc

In [ ]:
# Find dates
dates_str = [d for d in os.listdir(dirs.raw_rover) if os.path.isdir(os.path.join(dirs.raw_rover, d))]
dates_str.sort()
print(dates_str)

In [ ]:
# Limit to a subset of dates for testing
dates_str = [date for date in dates_str if date < "2022-07-01"]

In [ ]:
dir_nb1 = ROOT / 'notebooks_data'
dir_nb2 = ROOT / 'notebooks_process'
dir_out = ROOT / 'notebooks_out' / farm_name
dir_out.mkdir(parents=True, exist_ok=True)

In [ ]:
sorted([f for f in os.listdir(dir_nb1) if f.endswith('.ipynb')])

In [ ]:
sorted([f for f in os.listdir(dir_nb2) if f.endswith('.ipynb')])

## Run the kriging notebook for all dates and modes
---

### Pre-process rover and soil data

In [ ]:
### Create canvas
if re_preprocess:

    notebook = '11_createGrids.ipynb'

    for grid_spacing, mode in [(100, ''), (200, 'test')]:
        pm.execute_notebook(dir_nb2 / notebook, 
                            dir_out /  notebook.replace('.ipynb', f'_{mode}.ipynb'),
                            parameters=dict(grid_spacing=grid_spacing, mode=mode)
                            )


In [ ]:
if re_preprocess:
    for notebook in ['12_roverProcess.ipynb', '13_sampleWeight.ipynb']:
        try:
            pm.execute_notebook(
                    dir_nb2 / notebook, 
                    dir_out / notebook,
                    )
        except Exception as e:
            print(f"Error executing {notebook}: {e}")

### Interpolate soil properties (Ordinary Kriging)

In [ ]:
if re_preprocess:
    notebook = '14_sampleKrige_ok.ipynb'
    try:
        pm.execute_notebook(
            dir_nb2 / notebook, 
            dir_out / notebook
        )
    except Exception as e:
        print(f"Error executing {notebook}: {e}")

In [ ]:
#  Krige rover data for all dates

if re_preprocess:
    notebook = '15_corrCounts.ipynb'
    for date_str in dates_str:
        try:
            pm.execute_notebook(
                dir_nb2 / notebook, 
                dir_out /notebook.replace('.ipynb', f'_{date_str}.ipynb'),
                parameters=dict(date_str=date_str)
        )
        except Exception as e:
            print(f"Error processing {date_str}: {e}")

### Corrected Counts Kriging (B2PK): Fitting
The purpose of testing run:  
    1. Fit variogram.  
    2. Ensure the workflow runs successfully, the matrices are solvable, and the results are reasonable.  
    3. Quick train-split validation.  
    4. The testing run only uses a small subset of the dataset (~1/10) for fast excution and performs prediction on a coarse-resolution (e.g., 300 m) grid.

During the testing phase, the kriging parameters (e.g., variogram model type, nugget, sill, range, anisotropy) must be manually adjusted in the output notebook to ensure a good fit.  
Once the parameters are finalized, the formal run will automatically use the variogram configuration produced in the test run as the input for operational processing.  
Note that the P2PK and B2PK have different variogram parameters.  

In [ ]:
if False:
    notebook = '16_roverKrige_b2pk.ipynb'
    for date_str in dates_str[:]:
        for mode in ['b2pk fit', 'p2pk fit']:
            try:
                pm.execute_notebook(
                    dir_nb2 / notebook, 
                    dir_out / notebook.replace('.ipynb', f'_{date_str}_{mode}.ipynb'),
                    parameters=dict(date_str=date_str, mode=mode)
                )
            except Exception as e:
                print(f"Error processing {date_str} with mode {mode}: {e}")
    

    sys.exit('Stopping after executing B2PK fitting notebooks')

### Corrected Counts Kriging (B2PK):Formal run

The formal run:  
    1. Load variagrom function fitted in the test run.  
    2. The formal run uses a subset of the  dataset whose mutual distance are greater than a minimum distaence. All remained data are used for testing.  
    3. It preforms prediction on a fine-resolution (e.g., 30 m) grid, as well as on soil moisture sampling locations for CNRS calibration.  

Note: The user may manually save the variogram parameter file `*_variogram.json` by adding a suffix `_saved` to its file name. If such a file exists, the formal run will automatically use the `_saved` version. The user must manually remove or rename this file when it is no longer needed.  

<!-- Amaroo_2021-04-28_CountsCorr_p2p_variogram.json -->

In [ ]:
#  Krige rover data for all dates
if True:
    notebook = '16_roverKrige_b2pk.ipynb'
    for date_str in dates_str[:]: 
        for mode in ['b2pk', 'p2pk']: # 'b2pk', 'p2pk' 
            try:
                pm.execute_notebook(
                    dir_nb2 /notebook, 
                    dir_out/ notebook.replace('.ipynb', f'_{date_str}_{mode}.ipynb'),
                    parameters=dict(date_str=date_str, mode=mode)
                )
            except Exception as e:
                print(f"Error processing {date_str} with mode {mode}: {e}")

    ##  Shutdown the computer in 5 minutes if uncommenting the following line.
    # ! shutdown /s /t 300 /c "Shutting down in 300 seconds as the data processing pipeline has completed. Run the following command to cancel the shutdown: shutdown /a"

### Calibrate

In [ ]:
# Calibrate N0. Clean mode.
if re_calibrate:
    notebook = '17_calN0.ipynb'
    for mode in ['b2pk', 'p2pk']: # 'b2pk', 'p2pk'
        pm.execute_notebook(
            dir_nb2 / notebook,
            dir_out / notebook.replace('.ipynb', f'_{mode}_clean.ipynb'),
            parameters=dict(mode=mode, remove_anomalies=True)
        )


### Map soil moisture

In [ ]:
# Map Soil Moisture
if re_calibrate:
    notebook = '21_convertSM.ipynb'
    for mode in ['b2pk', 'p2pk']: # 'b2pk', 'p2pk'
        for date_str in dates_str[:]:
            pm.execute_notebook(
                dir_nb2 / notebook, 
                dir_out / notebook.replace('.ipynb', f'_{date_str}_{mode}.ipynb'),
                parameters=dict(date_str=date_str, mode=mode)
            )
